### Textual Analysis Pipeline Execution

This notebook serves as the execution interface for the textual analysis pipeline.
It imports the modular scripts developed for the project, including functions for
transcript processing, metadata extraction, and section parsing. The notebook first
adds the `program/scripts` directory to the Python path to ensure the custom modules
can be imported. It then loads the main pipeline functions required to process the
earnings call transcripts and generate the final dataset containing AI term counts,
LM sentiment metrics, and transcript metadata.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
SCRIPTS_DIR = PROJECT_ROOT / "program" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

Import scripts

In [2]:
from pipeline import run_corpus, process_transcript
from metadata import extract_call_date, extract_fiscal_quarter_year
from text_processing import split_presentation_and_qa

Execute codes

In [3]:
df = run_corpus(
    input_root="../../data/raw/transcript",
    out_csv="../../data/processed/textual_analysis_results.csv",
    out_xlsx="../../data/processed/textual_analysis_results.xlsx",
    lm_dict_path="../../data/raw/Loughran-McDonald_MasterDictionary_1993-2024.csv",
)

df.head()

,company,ticker,date,file,total_words,pres_words,qa_words,core_hits_total,core_hits_pres,core_hits_qa,...,lm_uncertainty_qa,fiscal_quarter,fiscal_year,fiscal_period,core_per_1000_total,core_per_1000_pres,core_per_1000_qa,adj_per_1000_total,adj_per_1000_pres,adj_per_1000_qa
0,Alphabet Inc,GOOGL,2022-04-26,2022-Apr-26-GOOGL.OQ-138254363287-Transcript.txt,8430,4250,4020,9,9,0,...,0.009904,1,2022,Q1 2022,1.067616,2.117647,0.000000,1.423488,1.882353,0.995025
1,Alphabet Inc,GOOGL,2022-02-01,2022-Feb-01-GOOGL.OQ-138419589371-Transcript.txt,9563,4214,5189,32,16,16,...,0.008415,4,2021,Q4 2021,3.346230,3.796868,3.083446,1.359406,1.661130,1.156292
2,Alphabet Inc,GOOGL,2022-07-26,2022-Jul-26-GOOGL.OQ-140228385298-Transcript.txt,8594,3955,4468,22,14,8,...,0.013164,2,2022,Q2 2022,2.559926,3.539823,1.790510,0.814522,1.264223,0.447628
3,Alphabet Inc,GOOGL,2022-10-25,2022-Oct-25-GOOGL.OQ-139231849236-Transcript.txt,8954,4277,4511,20,11,9,...,0.010760,3,2022,Q3 2022,2.233639,2.571896,1.995123,1.116819,1.402852,0.886721
4,Alphabet Inc,GOOGL,2023-04-25,2023-Apr-25-GOOGL.OQ-140631444702-Transcript.txt,9229,4418,4649,93,64,29,...,0.010899,1,2023,Q1 2023,10.076931,14.486193,6.237901,1.191895,1.584427,0.860400


Final inspection / Sanity checks

In [5]:
# ================================
# Final Inspection / Sanity Checks
# ================================

import pandas as pd

issues = []

# 1. LM tone should be between -1 and 1
lm_tone_cols = ["lm_tone_total", "lm_tone_pres", "lm_tone_qa"]

for col in lm_tone_cols:
    bad = df[(df[col] < -1) | (df[col] > 1)]
    if not bad.empty:
        print(f"\n⚠️ LM tone out of bounds in {col}:")
        display(bad[["company","ticker","file_name",col]])
        issues.append(col)

# 2. LM uncertainty should be between 0 and 1
lm_unc_cols = ["lm_uncertainty_total", "lm_uncertainty_pres", "lm_uncertainty_qa"]

for col in lm_unc_cols:
    bad = df[(df[col] < 0) | (df[col] > 1)]
    if not bad.empty:
        print(f"\n⚠️ LM uncertainty out of bounds in {col}:")
        display(bad[["company","ticker","file_name",col]])
        issues.append(col)

# 3. Presentation section not detected (0 words)
bad_pres = df[df["pres_words"] == 0]

if not bad_pres.empty:
    print("\n⚠️ Presentation section not detected (0 words):")
    display(bad_pres[["company","ticker","file_name","words_pres"]])
    issues.append("presentation_zero")

# 4. Q&A section not detected (0 words)
bad_qa = df[df["qa_words"] == 0]

if not bad_qa.empty:
    print("\n⚠️ Q&A section not detected (0 words):")
    display(bad_qa[["company","ticker","file_name","words_qa"]])
    issues.append("qa_zero")

# Final summary
if not issues:
    print("✅ All sanity checks passed. No anomalies detected.")
else:
    print(f"\nInspection completed. Issues detected in: {set(issues)}")

✅ All sanity checks passed. No anomalies detected.
